# Cross-topic Argument Mining with RoBERTa — Colab (end-to-end)

Re-implementation of Stab et al. (EMNLP 2018), *Cross-topic Argument Mining from Heterogeneous Sources*, with **RoBERTa replacing the Contextual BiLSTM (BiCLSTM)**.

Run the cells top-to-bottom. The only cell you need to edit is **Section C (Configuration)** — paths, hyperparameters, and seeds.

**Sections**
- A. Mount Drive
- B. Install dependencies
- C. **Configuration** (edit me)
- D. Imports
- E. Data — UKP loader
- F. Data — DIP2016 loader
- G. Tokenization & collation
- H. Model — single-task RoBERTa
- I. Model — RoBERTa MTL (shared encoder + UKP head + DIP head)
- J. Metrics (Table 4 of the paper)
- K. Training routines (single-task + MTL, with checkpointing & loss tracking)
- L. Sanity check
- M. Smoke test (1 topic, 1 seed, 2 epochs)
- N. Full grid (resumable, writes per-run JSONs to Drive)
- O. Inspect summary
- P. Run status board — which datasets / seeds done, in-flight, or not started
- Q. Confusion matrices — per-dataset and overall, both 2- and 3-class
- R. Train / val loss curves — average overfitting / underfitting view per setup
- S. Classification reports — sklearn precision/recall/F1 per dataset and overall
- **T. Paper-style metric tables — 2-class (F1, P_arg, R_arg) & 3-class (F1, P_arg±, R_arg±)**

## A. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## B. Install dependencies

In [ ]:
!pip install -q "transformers>=4.41" "scikit-learn>=1.3" "pandas>=2.0" "tqdm>=4.66"

## C. Configuration  ← edit me

Every path, hyperparameter, and seed lives here.  Nothing below this cell needs editing for normal runs.

In [ ]:
from pathlib import Path

# ----- Paths on Google Drive (absolute) -----
UKP_CSV    = Path('/content/drive/MyDrive/PhD Ali 26/Dataset/UKP csv/8 UKP datasets.csv')
DIP_DIR    = Path('/content/drive/MyDrive/PhD Ali 26/Dataset/DIP2016')
OUTPUT_DIR = Path('/content/drive/MyDrive/PhD Ali 26/results/Ro2026')

# Optional: CSV with columns [queryID, query_text] for DIP MTL.
# Leave as None if you don't have one (queryID itself will be used as text).
QUERY_TEXT_CSV: Path | None = None

# ----- Model / tokenization -----
MODEL_NAME = 'roberta-base'
MAX_LENGTH = 128

# ----- Optimization -----
EPOCHS           = 10
BATCH_SIZE       = 32
EVAL_BATCH_SIZE  = 64
LR               = 2e-5
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.06
GRAD_CLIP        = 1.0
NUM_WORKERS      = 0   # 0 silences Colab's noisy DataLoader-worker shutdown traces

# ----- Experimental protocol -----
SEEDS         = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]   # paper uses 10 seeds
LABEL_SETUPS  = [2, 3]                            # 2-label and 3-label

# Section N runs BOTH variants below for every (topic, seed):
#   True   -> MTL with DIP2016     (replaces the paper's `mtl+biclstm+dip2016`)
#   False  -> single-task RoBERTa  (no DIP2016, replaces the paper's `biclstm`)
# Order matters: the first entry is trained first. Drop one entry if you only want one variant.
MTL_MODES     = [True, False]   # MTL+DIP2016 first; single-task (no DIP) second

TEST_TOPICS: list[str] | None = None              # None = all 8 paper topics; or e.g. ['gun control']

# ----- MTL specifics -----
DIP_MAX_EXAMPLES = 300_000   # paper: 300K of 600K. Set None to use all.

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert UKP_CSV.exists(),  f'UKP file not found: {UKP_CSV}'
assert DIP_DIR.exists(),  f'DIP folder not found: {DIP_DIR}'
print('UKP   :', UKP_CSV)
print('DIP   :', DIP_DIR)
print('OUT   :', OUTPUT_DIR)
print('Modes :', ['MTL + DIP2016' if m else 'single-task (no DIP)' for m in MTL_MODES])


## D. Imports

In [ ]:
import json
import random
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field, replace
from functools import partial
from typing import Iterable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, precision_recall_fscore_support
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    RobertaForSequenceClassification,
    RobertaModel,
    RobertaTokenizerFast,
    get_linear_schedule_with_warmup,
)

TOPICS = [
    'abortion', 'cloning', 'death penalty', 'gun control',
    'marijuana legalization', 'minimum wage', 'nuclear energy', 'school uniforms',
]
LABELS_3 = ['NoArgument', 'Argument_against', 'Argument_for']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## E. Data — UKP loader

UKP is a single CSV with columns:
`topic, retrievedUrl, archivedUrl, sentenceHash, sentence, annotation, set` where
`annotation ∈ {NoArgument, Argument_for, Argument_against}` and
`set ∈ {train, val, test}`.

In [ ]:
@dataclass
class Example:
    topic: str
    sentence: str
    label: int

def label_to_id(annotation: str, num_labels: int) -> int:
    if num_labels == 3:
        return LABELS_3.index(annotation)
    return 0 if annotation == 'NoArgument' else 1

def load_ukp_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    missing = {'topic', 'sentence', 'annotation', 'set'} - set(df.columns)
    if missing:
        raise ValueError(f'UKP CSV missing columns: {missing}')
    df['topic_norm'] = df['topic'].astype(str).str.strip().str.lower()
    return df

def build_ukp_splits(csv_path: Path, test_topic: str, num_labels: int) -> dict[str, list[Example]]:
    df = load_ukp_csv(csv_path)
    t = test_topic.strip().lower()
    if t not in df['topic_norm'].unique():
        raise ValueError(f'Topic {test_topic!r} not in CSV. Available: {sorted(df["topic_norm"].unique())}')
    train, val, test = [], [], []
    for _, row in df.iterrows():
        ex = Example(
            topic=str(row['topic']),
            sentence=str(row['sentence']),
            label=label_to_id(str(row['annotation']), num_labels),
        )
        if row['topic_norm'] == t:
            if row['set'] == 'test':
                test.append(ex)
        else:
            if row['set'] == 'train':
                train.append(ex)
            elif row['set'] == 'val':
                val.append(ex)
    return {'train': train, 'val': val, 'test': test}

class UKPDataset(Dataset):
    def __init__(self, examples: Iterable[Example], tokenizer, max_length: int = 128):
        self.examples = list(examples)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self) -> int:
        return len(self.examples)
    def __getitem__(self, idx: int) -> dict:
        ex = self.examples[idx]
        enc = self.tokenizer(ex.topic, ex.sentence,
                             truncation=True, max_length=self.max_length, padding=False)
        item = {k: torch.as_tensor(v) for k, v in enc.items()}
        item['labels'] = torch.as_tensor(ex.label, dtype=torch.long)
        return item

## F. Data — DIP2016 loader

Parses XML files of shape `<singleQueryResults queryID=…><documents><document><sentences><s relevant="true|false"><content>…`.
Uses the `queryID` itself as the topic text when no `queryID → query_text` mapping is supplied.

In [ ]:
@dataclass
class DIPExample:
    query: str
    sentence: str
    label: int  # 1 = relevant, 0 = not relevant

def _parse_dip_xml(path: Path, query_text_map: dict[str, str] | None = None) -> list[DIPExample]:
    root = ET.parse(path).getroot()
    qid = root.attrib.get('queryID', path.stem)
    qtext = (query_text_map.get(str(qid)) if query_text_map else None) or str(qid)
    out: list[DIPExample] = []
    for s in root.iter('s'):
        rel = s.attrib.get('relevant', 'false').strip().lower()
        c = s.find('content')
        if c is None or c.text is None:
            continue
        text = c.text.strip()
        if not text:
            continue
        out.append(DIPExample(query=qtext, sentence=text, label=(1 if rel == 'true' else 0)))
    return out

def load_dip2016(dip_dir: Path,
                 query_text_map: dict[str, str] | None = None,
                 limit_files: int | None = None,
                 max_examples: int | None = None) -> list[DIPExample]:
    files = sorted(dip_dir.glob('*.xml'))
    if limit_files is not None:
        files = files[:limit_files]
    out: list[DIPExample] = []
    for f in files:
        out.extend(_parse_dip_xml(f, query_text_map))
        if max_examples is not None and len(out) >= max_examples:
            return out[:max_examples]
    return out

class DIPDataset(Dataset):
    def __init__(self, examples: Iterable[DIPExample], tokenizer, max_length: int = 128):
        self.examples = list(examples)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self) -> int:
        return len(self.examples)
    def __getitem__(self, idx: int) -> dict:
        ex = self.examples[idx]
        enc = self.tokenizer(ex.query, ex.sentence,
                             truncation=True, max_length=self.max_length, padding=False)
        item = {k: torch.as_tensor(v) for k, v in enc.items()}
        item['labels'] = torch.as_tensor(ex.label, dtype=torch.long)
        return item

# Optional queryID -> query_text map
QUERY_TEXT_MAP: dict[str, str] = {}
if QUERY_TEXT_CSV is not None and Path(QUERY_TEXT_CSV).exists():
    qdf = pd.read_csv(QUERY_TEXT_CSV)
    QUERY_TEXT_MAP = {str(q): str(t) for q, t in zip(qdf['queryID'], qdf['query_text'])}
    print(f'Loaded {len(QUERY_TEXT_MAP)} queryID->text mappings')
else:
    print('No queryID->text mapping; using queryID as text for DIP.')

## G. Collation

In [ ]:
def collate(batch: list[dict], pad_token_id: int) -> dict:
    max_len = max(x['input_ids'].size(0) for x in batch)
    out = {}
    for key in ('input_ids', 'attention_mask'):
        if key not in batch[0]:
            continue
        pad_val = pad_token_id if key == 'input_ids' else 0
        stacked = torch.full((len(batch), max_len), pad_val, dtype=torch.long)
        for i, x in enumerate(batch):
            stacked[i, :x[key].size(0)] = x[key]
        out[key] = stacked
    out['labels'] = torch.stack([x['labels'] for x in batch])
    return out

## H. Model — single-task RoBERTa

Topic information is injected by encoding `(topic, sentence)` as a RoBERTa sentence pair — the transformer analogue of the paper's i-/c-gate topic injection in the BiCLSTM.

In [ ]:
def build_single_task_model(model_name: str, num_labels: int):
    tokenizer = RobertaTokenizerFast.from_pretrained(model_name)
    model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    return model, tokenizer

## I. Model — RoBERTa MTL  (shared encoder + UKP head + DIP head)

Replaces `mtl+biclstm+dip2016` from the paper.

In [ ]:
class RobertaMTL(nn.Module):
    def __init__(self, model_name: str, main_num_labels: int,
                 aux_num_labels: int = 2, dropout: float = 0.1):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.main_head = nn.Linear(hidden, main_num_labels)
        self.aux_head  = nn.Linear(hidden, aux_num_labels)
        self.loss_fct  = nn.CrossEntropyLoss()
    def forward(self, input_ids, attention_mask, labels=None, task: str = 'main'):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(out.last_hidden_state[:, 0, :])  # <s> token
        head = self.main_head if task == 'main' else self.aux_head
        logits = head(pooled)
        loss = self.loss_fct(logits, labels) if labels is not None else None
        return {'loss': loss, 'logits': logits}

def build_mtl_model(model_name: str, main_num_labels: int):
    tokenizer = RobertaTokenizerFast.from_pretrained(model_name)
    model = RobertaMTL(model_name, main_num_labels=main_num_labels)
    return model, tokenizer

## J. Metrics  (Table 4 of Stab et al.)

- Macro-F1
- 2-label: `P_arg`, `R_arg`
- 3-label: `P_arg+`, `R_arg+`, `P_arg-`, `R_arg-`

In [ ]:
def compute_metrics(y_true, y_pred, num_labels: int) -> dict:
    out: dict[str, float] = {}
    out['macro_f1'] = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
    if num_labels == 2:
        p, r, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=[1], zero_division=0)
        out['P_arg'] = float(p[0]); out['R_arg'] = float(r[0])
    else:
        labels = [LABELS_3.index('Argument_for'), LABELS_3.index('Argument_against')]
        p, r, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
        out['P_arg+'] = float(p[0]); out['R_arg+'] = float(r[0])
        out['P_arg-'] = float(p[1]); out['R_arg-'] = float(r[1])
    return out

# Keys that are identifiers / per-run artifacts, NOT metrics. They must
# be excluded from aggregate_runs() so summary.json only contains real
# metrics (e.g. macro_f1, P_arg, ...), not noise like mean of 'seed'.
_NON_METRIC_KEYS = {
    'test_topic', 'seed', 'num_labels', 'use_mtl', 'model_name',
    'train_losses', 'val_losses', 'y_true', 'y_pred',
}

def aggregate_runs(runs: list[dict]) -> dict:
    keys = [
        k for k, v in runs[0].items()
        if k not in _NON_METRIC_KEYS
        and isinstance(v, (int, float))
        and not isinstance(v, bool)
    ]
    return {k: (float(np.mean([r[k] for r in runs])),
                float(np.std([r[k] for r in runs]))) for k in keys}

## K. Training routines

Per-epoch checkpoints are written to `OUTPUT_DIR/checkpoints/<tag>/ckpt.pt` and overwritten in place (single slot). If Colab disconnects mid-run, just re-execute Section N — the in-flight run resumes at the next epoch with its optimizer, scheduler, and RNG state restored; runs whose final JSON already exists are skipped entirely. Each ckpt is ~1–1.5 GB and is deleted once the run completes and the final JSON has been written.

In [ ]:
@dataclass
class HParams:
    ukp_csv: Path = UKP_CSV
    dip_dir: Path = DIP_DIR
    output_dir: Path = OUTPUT_DIR
    model_name: str = MODEL_NAME
    max_length: int = MAX_LENGTH
    epochs: int = EPOCHS
    batch_size: int = BATCH_SIZE
    eval_batch_size: int = EVAL_BATCH_SIZE
    lr: float = LR
    weight_decay: float = WEIGHT_DECAY
    warmup_ratio: float = WARMUP_RATIO
    grad_clip: float = GRAD_CLIP
    num_workers: int = NUM_WORKERS
    num_labels: int = 2
    test_topic: str = 'gun control'
    seed: int = 0
    use_mtl: bool = False
    dip_max_examples: int | None = DIP_MAX_EXAMPLES
    dip_query_text_map: dict[str, str] = field(default_factory=lambda: QUERY_TEXT_MAP)

def run_tag(hp: HParams) -> str:
    mtl = 'mtl' if hp.use_mtl else 'single'
    return f"{mtl}_L{hp.num_labels}_{hp.test_topic.replace(' ', '_')}_seed{hp.seed}"

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def _make_optim(model: nn.Module, hp: HParams, total_steps: int):
    no_decay = ('bias', 'LayerNorm.weight')
    groups = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': hp.weight_decay},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    optim = AdamW(groups, lr=hp.lr)
    sched = get_linear_schedule_with_warmup(optim, int(total_steps * hp.warmup_ratio), total_steps)
    return optim, sched

# --- Checkpointing ---------------------------------------------------------
# After every epoch we overwrite a single ckpt.pt with model+optim+sched+epoch
# +best-so-far + train/val loss history. On restart, the run resumes at
# epoch+1 with the best weights restored. The ckpt is ~1-1.5 GB
# (RoBERTa-base + AdamW state) and is deleted once the run finishes
# successfully and the final JSON is on Drive.

def _ckpt_path(hp: HParams) -> Path:
    return hp.output_dir / 'checkpoints' / run_tag(hp) / 'ckpt.pt'

def _save_ckpt(hp, model, optim, sched, epoch, best_val, best_state,
               train_losses, val_losses):
    path = _ckpt_path(hp)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.pt.tmp')
    torch.save({
        'model': model.state_dict(),
        'optim': optim.state_dict(),
        'sched': sched.state_dict(),
        'epoch': epoch,
        'best_val_loss': best_val,
        'best_state': best_state,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'rng_torch': torch.get_rng_state(),
        'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        'rng_numpy': np.random.get_state(),
        'rng_python': random.getstate(),
    }, tmp)
    tmp.replace(path)  # atomic rename: a crash mid-save never corrupts ckpt

def _load_ckpt(hp, model, optim, sched):
    path = _ckpt_path(hp)
    if not path.exists():
        return 0, float('inf'), None, [], []
    # weights_only=False: ckpt contains NumPy/Python RNG state which the
    # PyTorch 2.6 safe-loader refuses. The file is one we wrote ourselves.
    data = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(data['model'])
    optim.load_state_dict(data['optim'])
    sched.load_state_dict(data['sched'])
    torch.set_rng_state(data['rng_torch'].cpu())
    if data.get('rng_cuda') is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in data['rng_cuda']])
    np.random.set_state(data['rng_numpy'])
    random.setstate(data['rng_python'])
    start = data['epoch'] + 1
    print(f"  resuming {run_tag(hp)}: {start}/{hp.epochs} epochs done, "
          f"{hp.epochs - start} remaining (best val loss so far {data['best_val_loss']:.4f})")
    return (start, data['best_val_loss'], data['best_state'],
            data.get('train_losses', []), data.get('val_losses', []))

def _ckpt_epoch(hp: HParams) -> int | None:
    """Number of completed epochs in the in-flight ckpt (None if no ckpt)."""
    path = _ckpt_path(hp)
    if not path.exists():
        return None
    try:
        data = torch.load(path, map_location='cpu', weights_only=False)
        return data['epoch'] + 1
    except Exception:
        return None

def _cleanup_ckpt(hp: HParams) -> None:
    path = _ckpt_path(hp)
    if path.exists():
        path.unlink()
    try:
        path.parent.rmdir()
    except OSError:
        pass

def _epoch_bar(hp: HParams, start_epoch: int):
    """tqdm over epochs, pre-filled with whatever was already completed."""
    return tqdm(
        total=hp.epochs,
        initial=start_epoch,
        desc=f'{run_tag(hp)} epochs',
        unit='ep',
        position=1,           # below the grid bar
        leave=True,
    )
# ---------------------------------------------------------------------------

@torch.no_grad()
def _evaluate(model, loader, mtl_task: str | None = None):
    model.eval()
    losses, preds, golds = [], [], []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        if mtl_task is None:
            out = model(**batch); loss, logits = out.loss, out.logits
        else:
            out = model(**batch, task=mtl_task); loss, logits = out['loss'], out['logits']
        losses.append(loss.item() * batch['labels'].size(0))
        preds.append(logits.argmax(dim=-1).cpu().numpy())
        golds.append(batch['labels'].cpu().numpy())
    n = sum(len(g) for g in golds)
    return sum(losses) / max(n, 1), np.concatenate(preds), np.concatenate(golds)

def _train_one_epoch(model, loader, optim, sched, hp, task: str | None = None,
                     pbar_desc: str = 'train'):
    """Runs one training pass. Returns sample-weighted average loss."""
    model.train()
    pbar = tqdm(loader, desc=pbar_desc, position=2, leave=False)
    loss_sum, n = 0.0, 0
    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        if task is None:
            out = model(**batch); loss = out.loss
        else:
            out = model(**batch, task=task); loss = out['loss']
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), hp.grad_clip)
        optim.step(); sched.step(); optim.zero_grad()
        bsz = batch['labels'].size(0)
        loss_sum += float(loss.item()) * bsz; n += bsz
        pbar.set_postfix(loss=float(loss.item()))
    return loss_sum / max(n, 1)

def _train_single(hp: HParams) -> dict:
    splits = build_ukp_splits(hp.ukp_csv, hp.test_topic, hp.num_labels)
    model, tok = build_single_task_model(hp.model_name, hp.num_labels)
    model.to(DEVICE)
    coll = partial(collate, pad_token_id=tok.pad_token_id)
    train_loader = DataLoader(UKPDataset(splits['train'], tok, hp.max_length),
                              batch_size=hp.batch_size, shuffle=True,
                              collate_fn=coll, num_workers=hp.num_workers)
    val_loader = DataLoader(UKPDataset(splits['val'], tok, hp.max_length),
                            batch_size=hp.eval_batch_size, collate_fn=coll)
    test_loader = DataLoader(UKPDataset(splits['test'], tok, hp.max_length),
                             batch_size=hp.eval_batch_size, collate_fn=coll)
    optim, sched = _make_optim(model, hp, len(train_loader) * hp.epochs)
    start_epoch, best_val, best_state, train_losses, val_losses = \
        _load_ckpt(hp, model, optim, sched)
    ebar = _epoch_bar(hp, start_epoch)
    for epoch in range(start_epoch, hp.epochs):
        train_loss = _train_one_epoch(model, train_loader, optim, sched, hp,
                                      pbar_desc=f'ep {epoch+1}/{hp.epochs}')
        val_loss, _, _ = _evaluate(model, val_loader)
        train_losses.append(train_loss); val_losses.append(val_loss)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        _save_ckpt(hp, model, optim, sched, epoch, best_val, best_state,
                   train_losses, val_losses)
        ebar.set_postfix(train=f'{train_loss:.4f}', val=f'{val_loss:.4f}',
                         best=f'{best_val:.4f}')
        ebar.update(1)
    ebar.close()
    if best_state is not None:
        model.load_state_dict(best_state)
    _, preds, golds = _evaluate(model, test_loader)
    return (compute_metrics(golds, preds, hp.num_labels)
            | {'best_val_loss': best_val,
               'train_losses': train_losses,
               'val_losses': val_losses,
               'y_true': golds.tolist(),
               'y_pred': preds.tolist()})

def _train_mtl(hp: HParams) -> dict:
    splits = build_ukp_splits(hp.ukp_csv, hp.test_topic, hp.num_labels)
    model, tok = build_mtl_model(hp.model_name, hp.num_labels)
    model.to(DEVICE)
    coll = partial(collate, pad_token_id=tok.pad_token_id)
    train_loader = DataLoader(UKPDataset(splits['train'], tok, hp.max_length),
                              batch_size=hp.batch_size, shuffle=True,
                              collate_fn=coll, num_workers=hp.num_workers)
    val_loader = DataLoader(UKPDataset(splits['val'], tok, hp.max_length),
                            batch_size=hp.eval_batch_size, collate_fn=coll)
    test_loader = DataLoader(UKPDataset(splits['test'], tok, hp.max_length),
                             batch_size=hp.eval_batch_size, collate_fn=coll)
    dip_examples = load_dip2016(hp.dip_dir,
                                query_text_map=hp.dip_query_text_map or None,
                                max_examples=hp.dip_max_examples)
    aux_loader = DataLoader(DIPDataset(dip_examples, tok, hp.max_length),
                            batch_size=hp.batch_size, shuffle=True,
                            collate_fn=coll, num_workers=hp.num_workers)
    optim, sched = _make_optim(model, hp, (len(train_loader) + len(aux_loader)) * hp.epochs)
    start_epoch, best_val, best_state, train_losses, val_losses = \
        _load_ckpt(hp, model, optim, sched)
    ebar = _epoch_bar(hp, start_epoch)
    for epoch in range(start_epoch, hp.epochs):
        _train_one_epoch(model, aux_loader, optim, sched, hp, task='aux',
                         pbar_desc=f'ep {epoch+1}/{hp.epochs} [aux]')
        train_loss = _train_one_epoch(model, train_loader, optim, sched, hp,
                                      task='main',
                                      pbar_desc=f'ep {epoch+1}/{hp.epochs} [main]')
        val_loss, _, _ = _evaluate(model, val_loader, mtl_task='main')
        train_losses.append(train_loss); val_losses.append(val_loss)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        _save_ckpt(hp, model, optim, sched, epoch, best_val, best_state,
                   train_losses, val_losses)
        ebar.set_postfix(train=f'{train_loss:.4f}', val=f'{val_loss:.4f}',
                         best=f'{best_val:.4f}')
        ebar.update(1)
    ebar.close()
    if best_state is not None:
        model.load_state_dict(best_state)
    _, preds, golds = _evaluate(model, test_loader, mtl_task='main')
    return (compute_metrics(golds, preds, hp.num_labels)
            | {'best_val_loss': best_val,
               'train_losses': train_losses,
               'val_losses': val_losses,
               'y_true': golds.tolist(),
               'y_pred': preds.tolist()})

def run_one(hp: HParams) -> dict:
    # Only seed when starting fresh; a resumed run restores RNG from the ckpt.
    if not _ckpt_path(hp).exists():
        set_seed(hp.seed)
    metrics = _train_mtl(hp) if hp.use_mtl else _train_single(hp)
    metrics.update({
        'test_topic': hp.test_topic,
        'seed': hp.seed,
        'num_labels': hp.num_labels,
        'use_mtl': hp.use_mtl,
        'model_name': hp.model_name,
    })
    _cleanup_ckpt(hp)  # final JSON is the source of truth; drop the ~1GB ckpt
    return metrics

## L. Sanity check

In [ ]:
df = load_ukp_csv(UKP_CSV)
print('UKP rows:', len(df))
print('UKP topics:', sorted(df['topic_norm'].unique()))
print(df.groupby(['topic_norm', 'set', 'annotation']).size().unstack(fill_value=0).head(20))

dip_sample = load_dip2016(DIP_DIR, query_text_map=QUERY_TEXT_MAP or None, limit_files=3)
print(f'\nDIP (first 3 files): {len(dip_sample)} sentences')
for ex in dip_sample[:3]:
    print(f'  query={ex.query!r}  label={ex.label}  sent={ex.sentence[:80]!r}')

## M. Smoke test  (1 topic, 1 seed, 2 epochs, single-task)

~3–5 min on a T4 GPU. Confirms the pipeline runs end-to-end before launching the long grid.

In [ ]:
hp_smoke = HParams(epochs=2, test_topic='gun control', num_labels=2, seed=0, use_mtl=False)
smoke_res = run_one(hp_smoke)
print(smoke_res)

## N. Full grid  (resumable at run-level **and** epoch-level)

Iterates `{single, MTL} × {2-label, 3-label} × all topics × all seeds`.

- A finished run is saved to `OUTPUT_DIR/<tag>.json` and skipped on rerun.
- An in-flight run writes `OUTPUT_DIR/checkpoints/<tag>/ckpt.pt` after each epoch. If Colab dies mid-epoch, just rerun this cell — the run picks up at the next epoch with optimizer / scheduler / RNG restored.

⚠️ The full grid (10 seeds × 8 topics × 2 setups × 2 modes = 320 runs) takes days on a single T4. Trim `SEEDS`, `LABEL_SETUPS`, `MTL_MODES`, or `TEST_TOPICS` in **Section C** before launching.

In [ ]:
topics_to_run = TEST_TOPICS or TOPICS

print('Plan:')
print(f'  modes        : {["MTL + DIP2016" if m else "single-task" for m in MTL_MODES]}')
print(f'  label setups : {LABEL_SETUPS}')
print(f'  topics       : {topics_to_run}')
print(f'  seeds        : {SEEDS}')

# Deterministic outer-product order: MTL_MODES x LABEL_SETUPS x topics x seeds.
plan = [
    (use_mtl, num_labels, topic, seed)
    for use_mtl in MTL_MODES
    for num_labels in LABEL_SETUPS
    for topic in topics_to_run
    for seed in SEEDS
]
def _tag(use_mtl, num_labels, topic, seed):
    return f"{'mtl' if use_mtl else 'single'}_L{num_labels}_{topic.replace(' ', '_')}_seed{seed}"

done_at_start = sum(
    1 for combo in plan
    if (OUTPUT_DIR / f'{_tag(*combo)}.json').exists()
)
print(f'\nGrid: {len(plan)} runs total, {done_at_start} already on Drive, '
      f'{len(plan) - done_at_start} remaining.')

# grid_bar at top, epoch_bar in the middle (1), per-batch at the bottom (2).
grid_bar = tqdm(total=len(plan), initial=done_at_start,
                desc='grid', unit='run', position=0, leave=True)

# Track which (mode, num_labels, topic) datasets are fully done so we can
# print a tick the moment the last seed for that dataset completes.
topic_seeds_done: dict[tuple, set] = {}
ticked: set = set()

results: dict[tuple, list[dict]] = {}
for combo in plan:
    use_mtl, num_labels, topic, seed = combo
    tag = _tag(*combo)
    out_path = OUTPUT_DIR / f'{tag}.json'
    grid_bar.set_postfix_str(tag)
    if out_path.exists():
        res = json.loads(out_path.read_text())
    else:
        hp = HParams(test_topic=topic, num_labels=num_labels,
                     seed=seed, use_mtl=use_mtl)
        res = run_one(hp)
        out_path.write_text(json.dumps(res, indent=2))
        grid_bar.update(1)
    results.setdefault((use_mtl, num_labels, topic), []).append(res)

    # Per-dataset tick: print ✓ the first time all SEEDS finish for a topic.
    key = (use_mtl, num_labels, topic)
    topic_seeds_done.setdefault(key, set()).add(seed)
    if len(topic_seeds_done[key]) == len(SEEDS) and key not in ticked:
        ticked.add(key)
        mode_str = 'mtl+dip' if use_mtl else 'single  '
        tqdm.write(f'  ✓  {mode_str} / {num_labels}-label / {topic} '
                   f'(all {len(SEEDS)} seeds trained & tested)')
grid_bar.close()

# Aggregate per (mode, labels) and overall — deterministic key order.
summary: dict = {}
for use_mtl in MTL_MODES:
    for num_labels in LABEL_SETUPS:
        key = f"{'mtl' if use_mtl else 'single'}_{num_labels}label"
        per_topic = {t: results[(use_mtl, num_labels, t)] for t in topics_to_run
                     if (use_mtl, num_labels, t) in results}
        if not per_topic:
            continue
        summary[key] = {t: aggregate_runs(r) for t, r in per_topic.items()}
        all_runs_flat = [r for rs in per_topic.values() for r in rs]
        summary[key]['__overall__'] = aggregate_runs(all_runs_flat)

(OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2))
print('\nSaved summary to', OUTPUT_DIR / 'summary.json')

# Final dataset checklist.
print('\n=== Final dataset checklist ===')
for use_mtl in MTL_MODES:
    mode_label = 'MTL + DIP2016' if use_mtl else 'single-task (no DIP)'
    print(f'\n[{mode_label}]')
    for num_labels in LABEL_SETUPS:
        print(f'  {num_labels}-label:')
        for topic in topics_to_run:
            done = len(topic_seeds_done.get((use_mtl, num_labels, topic), set()))
            mark = '✓' if done == len(SEEDS) else f'{done}/{len(SEEDS)}'
            print(f'    {mark}  {topic}')


## O. Inspect summary

In [ ]:
FIGS_DIR   = OUTPUT_DIR / 'figures';  FIGS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = OUTPUT_DIR / 'tables';   TABLES_DIR.mkdir(parents=True, exist_ok=True)

summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())

# Print overall numbers per setup.
for setup, by_topic in summary.items():
    print(f'\n=== {setup} ===')
    overall = by_topic['__overall__']
    for k, (m, s) in overall.items():
        print(f'  {k:<14s} {m:.4f} ± {s:.4f}')

# 1) Long-form aggregated CSV: one row per (setup, scope, metric).
rows = []
for setup, by_topic in summary.items():
    for scope, metrics in by_topic.items():
        for metric, (mean, std) in metrics.items():
            rows.append({'setup': setup, 'scope': scope,
                         'metric': metric, 'mean': mean, 'std': std})
summary_csv = TABLES_DIR / 'summary.csv'
pd.DataFrame(rows).to_csv(summary_csv, index=False)
print(f'\nWrote {summary_csv}  ({len(rows)} rows)')

# 2) Raw per-run CSV: one row per completed run with every metric + identifiers.
run_rows = []
metric_keys = ['macro_f1', 'P_arg', 'R_arg', 'P_arg+', 'P_arg-', 'R_arg+', 'R_arg-',
               'best_val_loss']
for p in sorted(OUTPUT_DIR.glob('*_seed*.json')):
    try:
        r = json.loads(p.read_text())
    except Exception:
        continue
    row = {
        'tag':        p.stem,
        'mode':       'mtl' if r.get('use_mtl') else 'single',
        'num_labels': r.get('num_labels'),
        'test_topic': r.get('test_topic'),
        'seed':       r.get('seed'),
        'model_name': r.get('model_name'),
    }
    for k in metric_keys:
        row[k] = r.get(k)
    # Also include the last-epoch losses (useful for quick over/underfitting check).
    tl = r.get('train_losses') or []
    vl = r.get('val_losses')   or []
    row['final_train_loss'] = tl[-1] if tl else None
    row['final_val_loss']   = vl[-1] if vl else None
    row['n_epochs_logged']  = len(tl)
    run_rows.append(row)

if run_rows:
    all_runs_csv = TABLES_DIR / 'all_runs.csv'
    pd.DataFrame(run_rows).sort_values(['mode', 'num_labels', 'test_topic', 'seed']) \
                          .to_csv(all_runs_csv, index=False)
    print(f'Wrote {all_runs_csv}  ({len(run_rows)} runs)')
else:
    print('No completed runs found yet — all_runs.csv not written.')

## P. Run status board

Static view of the entire grid: which `(mode, labels, topic, seed)` combos are
**done**, **in-flight** (with how many epochs completed), or **not started**.
Re-runnable any time — including before launching Section N to see what's left.

In [ ]:
FIGS_DIR   = OUTPUT_DIR / 'figures';  FIGS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = OUTPUT_DIR / 'tables';   TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Each column cell is exactly this many characters wide so header and data line up.
CELL_W = 6
def _fmt_cell(s: str) -> str:
    return f'{s:^{CELL_W}}'

def _done_epochs_from_json(json_path: Path) -> int | None:
    """Read how many epochs a completed run actually executed."""
    try:
        r = json.loads(json_path.read_text())
        tl = r.get('train_losses')
        if isinstance(tl, list) and tl:
            return len(tl)
    except Exception:
        pass
    return None

def _scan_status():
    """Returns (mode, num_labels, topic) -> {seed: (state, epochs_done)}."""
    out = {}
    topics = TEST_TOPICS or TOPICS
    for use_mtl in MTL_MODES:
        for num_labels in LABEL_SETUPS:
            for topic in topics:
                seed_state = {}
                for seed in SEEDS:
                    hp = HParams(test_topic=topic, num_labels=num_labels,
                                 seed=seed, use_mtl=use_mtl)
                    json_path = OUTPUT_DIR / f'{run_tag(hp)}.json'
                    if json_path.exists():
                        n_ep = _done_epochs_from_json(json_path) or EPOCHS
                        seed_state[seed] = ('done', n_ep)
                    else:
                        ep = _ckpt_epoch(hp)
                        seed_state[seed] = ('in-flight', ep) if ep is not None else ('todo', 0)
                out[(use_mtl, num_labels, topic)] = seed_state
    return out

status = _scan_status()
csv_rows = []
header_seed_block = ' '.join(_fmt_cell(f'seed{s}') for s in SEEDS)
table_w = len('topic'.ljust(24)) + 3 + len(header_seed_block) + 3 + len('done/total')

for use_mtl in MTL_MODES:
    for num_labels in LABEL_SETUPS:
        mode_str = 'mtl' if use_mtl else 'single'
        setup = f'{mode_str}_{num_labels}label'
        print(f'\n=== {setup} ===')
        header = f'{"topic":<24s} | {header_seed_block} | done/total'
        print(header); print('-' * len(header))
        topic_done_total = 0
        for topic in (TEST_TOPICS or TOPICS):
            cells = []; topic_done = 0
            row = {'setup': setup, 'topic': topic}
            for seed in SEEDS:
                state, ep = status[(use_mtl, num_labels, topic)][seed]
                if state == 'done':
                    cells.append(_fmt_cell('✓')); topic_done += 1
                    row[f'seed_{seed}'] = 'done'
                elif state == 'in-flight':
                    cells.append(_fmt_cell(f'{ep}/{EPOCHS}'))
                    row[f'seed_{seed}'] = f'inflight_{ep}_of_{EPOCHS}'
                else:
                    cells.append(_fmt_cell('.'))
                    row[f'seed_{seed}'] = 'todo'
            row['done'] = topic_done
            row['total'] = len(SEEDS)
            csv_rows.append(row)
            print(f'{topic:<24s} | {" ".join(cells)} | {topic_done}/{len(SEEDS)}')
            topic_done_total += topic_done
        # TOTAL row: right-aligned to the done/total column.
        seed_block_w = len(header_seed_block)
        print(f'{"TOTAL":<24s} | {" " * seed_block_w} | '
              f'{topic_done_total}/{len(SEEDS) * len(TEST_TOPICS or TOPICS)}')

# Grand totals across all setups.
all_combos = [(m, n, t, s) for m in MTL_MODES for n in LABEL_SETUPS
              for t in (TEST_TOPICS or TOPICS) for s in SEEDS]
done = sum(1 for m, n, t, s in all_combos
           if (OUTPUT_DIR / f'{run_tag(HParams(test_topic=t, num_labels=n, seed=s, use_mtl=m))}.json').exists())
inflight = sum(1 for m, n, t, s in all_combos
               if _ckpt_epoch(HParams(test_topic=t, num_labels=n, seed=s, use_mtl=m)) is not None)
print(f'\nGrand total: {done}/{len(all_combos)} done, '
      f'{inflight} in-flight (ckpt on Drive), '
      f'{len(all_combos) - done - inflight} not started')
print('Legend:  ✓ = done   N/M = in-flight (epochs done/total)   . = not started')

status_csv = TABLES_DIR / 'status.csv'
pd.DataFrame(csv_rows).to_csv(status_csv, index=False)
print(f'\nWrote {status_csv}')

## Q. Confusion matrices

Per held-out topic and overall (concatenation of test sets), for both 2-label
and 3-label setups. Predictions across all seeds are pooled before computing
each matrix, so a `(8 topics × 10 seeds)`-strong dataset backs the overall
matrix per setup.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

FIGS_DIR   = OUTPUT_DIR / 'figures';  FIGS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = OUTPUT_DIR / 'tables';   TABLES_DIR.mkdir(parents=True, exist_ok=True)

def _label_names(num_labels: int) -> list[str]:
    return ['NoArg', 'Arg'] if num_labels == 2 else ['NoArg', 'Arg-', 'Arg+']

def _load_results():
    """Returns list[dict] of all per-run JSONs in OUTPUT_DIR."""
    out = []
    for p in sorted(OUTPUT_DIR.glob('*_seed*.json')):
        try:
            out.append(json.loads(p.read_text()))
        except Exception as e:
            print(f'  could not parse {p.name}: {e}')
    return out

ALL_RUNS = _load_results()
print(f'Loaded {len(ALL_RUNS)} run JSONs from {OUTPUT_DIR}')
ALL_RUNS_WITH_PREDS = [r for r in ALL_RUNS if 'y_true' in r and 'y_pred' in r]
print(f'  of which {len(ALL_RUNS_WITH_PREDS)} include test predictions '
      f'(older runs from before the update do not).')

def _plot_cm(ax, cm, names, title):
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha='right')
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title, fontsize=10)
    thresh = cm.max() / 2 if cm.max() > 0 else 1
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f'{cm[i, j]}', ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black', fontsize=8)

def _save_cm_csv(cm: np.ndarray, names: list[str], path: Path) -> None:
    df = pd.DataFrame(cm, index=[f'true_{n}' for n in names],
                          columns=[f'pred_{n}' for n in names])
    df.to_csv(path)

for use_mtl in MTL_MODES:
    for num_labels in LABEL_SETUPS:
        names = _label_names(num_labels)
        topics = TEST_TOPICS or TOPICS
        runs = [r for r in ALL_RUNS_WITH_PREDS
                if r.get('use_mtl') == use_mtl and r.get('num_labels') == num_labels]
        setup = f"{'mtl' if use_mtl else 'single'}_{num_labels}label"
        if not runs:
            print(f'(skip {setup}: no runs with predictions)')
            continue

        n_topics = len(topics)
        ncols = 4; nrows = (n_topics + ncols) // ncols  # +1 for the OVERALL panel
        fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
        axes = axes.flatten()
        all_y_true, all_y_pred = [], []
        for ax, topic in zip(axes, topics):
            yt, yp = [], []
            for r in runs:
                if r.get('test_topic') == topic:
                    yt.extend(r['y_true']); yp.extend(r['y_pred'])
            all_y_true.extend(yt); all_y_pred.extend(yp)
            if yt:
                cm = confusion_matrix(yt, yp, labels=list(range(num_labels)))
                _plot_cm(ax, cm, names, f'{topic}\n(n={len(yt)})')
                _save_cm_csv(cm, names, TABLES_DIR /
                             f"cm_{setup}_{topic.replace(' ', '_')}.csv")
            else:
                ax.set_visible(False)
        # OVERALL
        ax = axes[len(topics)] if len(topics) < len(axes) else axes[-1]
        if all_y_true:
            cm = confusion_matrix(all_y_true, all_y_pred, labels=list(range(num_labels)))
            _plot_cm(ax, cm, names, f'OVERALL\n(n={len(all_y_true)})')
            _save_cm_csv(cm, names, TABLES_DIR / f'cm_{setup}_OVERALL.csv')
        for ax in axes[len(topics) + 1:]:
            ax.set_visible(False)
        fig.suptitle(f"{'mtl' if use_mtl else 'single'}-task — {num_labels}-label confusion matrices",
                     fontsize=14)
        fig.tight_layout()
        png_path = FIGS_DIR / f'cm_{setup}.png'
        fig.savefig(png_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'  wrote {png_path}')

## R. Train / val loss curves  (overfitting & underfitting)

Per epoch, averaged across all completed runs of each setup. The shaded band
is ± 1 std across runs. A growing gap between the (decreasing) train curve
and a (rising) val curve indicates **overfitting**; both staying high
indicates **underfitting**.

In [ ]:
import matplotlib.pyplot as plt

FIGS_DIR   = OUTPUT_DIR / 'figures';  FIGS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = OUTPUT_DIR / 'tables';   TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Self-contained: re-load all run JSONs so this cell works even if Q wasn't run.
def _load_results():
    out = []
    for p in sorted(OUTPUT_DIR.glob('*_seed*.json')):
        try:
            out.append(json.loads(p.read_text()))
        except Exception:
            pass
    return out

ALL_RUNS = _load_results()
print(f'Loaded {len(ALL_RUNS)} run JSONs from {OUTPUT_DIR}')

def _stack_curves(curve_lists: list[list[float]]) -> tuple[np.ndarray, np.ndarray]:
    """Pad to the longest sequence length, return mean/std arrays."""
    if not curve_lists:
        return np.array([]), np.array([])
    L = max(len(c) for c in curve_lists)
    arr = np.full((len(curve_lists), L), np.nan)
    for i, c in enumerate(curve_lists):
        arr[i, :len(c)] = c
    return np.nanmean(arr, axis=0), np.nanstd(arr, axis=0)

setups = [(m, n) for m in MTL_MODES for n in LABEL_SETUPS]
fig, axes = plt.subplots(1, len(setups), figsize=(5 * len(setups), 4),
                         sharey=False, squeeze=False)
axes = axes.flatten()

curves_rows = []
for ax, (use_mtl, num_labels) in zip(axes, setups):
    setup = f"{'mtl' if use_mtl else 'single'}_{num_labels}label"
    runs = [r for r in ALL_RUNS
            if r.get('use_mtl') == use_mtl and r.get('num_labels') == num_labels
            and 'train_losses' in r and 'val_losses' in r]
    if not runs:
        ax.text(0.5, 0.5, 'no curves yet', ha='center', va='center',
                transform=ax.transAxes); ax.set_axis_off(); continue
    tr_mean, tr_std = _stack_curves([r['train_losses'] for r in runs])
    va_mean, va_std = _stack_curves([r['val_losses'] for r in runs])
    epochs_x = np.arange(1, len(tr_mean) + 1)
    ax.plot(epochs_x, tr_mean, label='train loss', color='tab:blue')
    ax.fill_between(epochs_x, tr_mean - tr_std, tr_mean + tr_std,
                    alpha=0.2, color='tab:blue')
    ax.plot(epochs_x, va_mean, label='val loss', color='tab:orange')
    ax.fill_between(epochs_x, va_mean - va_std, va_mean + va_std,
                    alpha=0.2, color='tab:orange')
    ax.set_xlabel('epoch'); ax.set_ylabel('cross-entropy loss')
    ax.set_title(f"{'mtl' if use_mtl else 'single'} — {num_labels}-label\n"
                 f'(avg of {len(runs)} runs)', fontsize=11)
    ax.legend(loc='best'); ax.grid(alpha=0.3)
    for i, ep in enumerate(epochs_x):
        curves_rows.append({
            'setup': setup, 'epoch': int(ep), 'n_runs': len(runs),
            'train_mean': float(tr_mean[i]), 'train_std': float(tr_std[i]),
            'val_mean':   float(va_mean[i]), 'val_std':   float(va_std[i]),
        })

fig.suptitle('Average train/val loss across runs (± 1 std band)', fontsize=14)
fig.tight_layout()
png_path = FIGS_DIR / 'loss_curves.png'
fig.savefig(png_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'wrote {png_path}')

if curves_rows:
    csv_path = TABLES_DIR / 'loss_curves.csv'
    pd.DataFrame(curves_rows).to_csv(csv_path, index=False)
    print(f'wrote {csv_path}  ({len(curves_rows)} rows)')

## S. Classification reports

`sklearn.metrics.classification_report` (precision, recall, F1, support) for
each held-out topic and overall, for every setup. Predictions are pooled
across seeds before scoring.

In [ ]:
from sklearn.metrics import classification_report

FIGS_DIR   = OUTPUT_DIR / 'figures';  FIGS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = OUTPUT_DIR / 'tables';   TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Self-contained: load run JSONs so this cell works even if Q wasn't run.
def _load_results():
    out = []
    for p in sorted(OUTPUT_DIR.glob('*_seed*.json')):
        try:
            out.append(json.loads(p.read_text()))
        except Exception:
            pass
    return out

ALL_RUNS = _load_results()
ALL_RUNS_WITH_PREDS = [r for r in ALL_RUNS if 'y_true' in r and 'y_pred' in r]
print(f'Loaded {len(ALL_RUNS)} run JSONs, {len(ALL_RUNS_WITH_PREDS)} with predictions.')

def _label_names(num_labels: int) -> list[str]:
    return ['NoArg', 'Arg'] if num_labels == 2 else ['NoArg', 'Arg-', 'Arg+']

reports_path = OUTPUT_DIR / 'classification_reports.txt'
text_lines: list[str] = []
combined_rows = []  # one row per (setup, scope, class)

def _report_to_rows(setup: str, scope: str, y_true, y_pred,
                    num_labels: int, names: list[str]) -> list[dict]:
    """Run sklearn classification_report in dict mode and flatten."""
    d = classification_report(y_true, y_pred, labels=list(range(num_labels)),
                              target_names=names, output_dict=True,
                              zero_division=0)
    rows = []
    for cls, vals in d.items():
        if isinstance(vals, dict):
            rows.append({'setup': setup, 'scope': scope, 'class': cls,
                         'precision': vals.get('precision'),
                         'recall':    vals.get('recall'),
                         'f1':        vals.get('f1-score'),
                         'support':   vals.get('support')})
        else:
            rows.append({'setup': setup, 'scope': scope, 'class': cls,
                         'precision': None, 'recall': None,
                         'f1': float(vals), 'support': None})
    return rows

for use_mtl in MTL_MODES:
    for num_labels in LABEL_SETUPS:
        names = _label_names(num_labels)
        topics = TEST_TOPICS or TOPICS
        runs = [r for r in ALL_RUNS_WITH_PREDS
                if r.get('use_mtl') == use_mtl and r.get('num_labels') == num_labels]
        if not runs:
            continue
        setup = f"{'mtl' if use_mtl else 'single'}_{num_labels}label"
        header = f"\n{'=' * 72}\n{setup.upper()}\n{'=' * 72}"
        text_lines.append(header); print(header)

        all_y_true, all_y_pred = [], []
        for topic in topics:
            yt, yp = [], []
            for r in runs:
                if r.get('test_topic') == topic:
                    yt.extend(r['y_true']); yp.extend(r['y_pred'])
            all_y_true.extend(yt); all_y_pred.extend(yp)
            if not yt:
                continue
            sub = f"\n--- {topic}  (n={len(yt)}) ---"
            rep = classification_report(yt, yp, labels=list(range(num_labels)),
                                        target_names=names, digits=4, zero_division=0)
            text_lines += [sub, rep]; print(sub); print(rep)
            rows = _report_to_rows(setup, topic, yt, yp, num_labels, names)
            combined_rows += rows
            pd.DataFrame(rows).to_csv(
                TABLES_DIR / f"cls_{setup}_{topic.replace(' ', '_')}.csv",
                index=False)

        if all_y_true:
            sub = f"\n--- OVERALL  (n={len(all_y_true)}) ---"
            rep = classification_report(all_y_true, all_y_pred,
                                        labels=list(range(num_labels)),
                                        target_names=names, digits=4, zero_division=0)
            text_lines += [sub, rep]; print(sub); print(rep)
            rows = _report_to_rows(setup, 'OVERALL', all_y_true, all_y_pred,
                                   num_labels, names)
            combined_rows += rows
            pd.DataFrame(rows).to_csv(TABLES_DIR / f'cls_{setup}_OVERALL.csv',
                                      index=False)

reports_path.write_text('\n'.join(text_lines))
combined_csv = TABLES_DIR / 'classification_reports.csv'
pd.DataFrame(combined_rows).to_csv(combined_csv, index=False)
print(f'\nFull text report: {reports_path}')
print(f'Combined CSV   : {combined_csv}  ({len(combined_rows)} rows)')

## T. Paper-style metric tables  (Table 4 of Stab et al.)

One table per `(mode, num_labels)`. Rows = held-out topics + OVERALL.
Columns mirror the paper's Table 4 exactly:

- **2-label**: `F1`, `P_arg`, `R_arg`
- **3-label**: `F1`, `P_arg+`, `P_arg-`, `R_arg+`, `R_arg-`

Each cell is `mean ± std` over `SEEDS`. CSVs are written alongside (one
with formatted strings, one with separate mean/std columns for downstream
plotting).

In [ ]:
TABLES_DIR = OUTPUT_DIR / 'tables';  TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Self-contained: load run JSONs so this cell works even if Q wasn't run.
def _load_results():
    out = []
    for p in sorted(OUTPUT_DIR.glob('*_seed*.json')):
        try:
            out.append(json.loads(p.read_text()))
        except Exception:
            pass
    return out

ALL_RUNS = _load_results()
print(f'Loaded {len(ALL_RUNS)} run JSONs')

# Column orderings, named exactly as in the paper. macro_f1 is the in-code
# key for the F1 column.
PAPER_COLS = {
    2: [('F1',      'macro_f1'),
        ('P_arg',   'P_arg'),
        ('R_arg',   'R_arg')],
    3: [('F1',      'macro_f1'),
        ('P_arg+',  'P_arg+'),
        ('P_arg-',  'P_arg-'),
        ('R_arg+',  'R_arg+'),
        ('R_arg-',  'R_arg-')],
}

def _agg(metric_key: str, runs: list[dict]) -> tuple[float, float] | tuple[None, None]:
    vals = [r[metric_key] for r in runs if metric_key in r]
    if not vals:
        return None, None
    return float(np.mean(vals)), float(np.std(vals))

topics = TEST_TOPICS or TOPICS
combined_rows = []  # for a single all-setups CSV

for use_mtl in MTL_MODES:
    for num_labels in LABEL_SETUPS:
        setup = f"{'mtl' if use_mtl else 'single'}_{num_labels}label"
        runs = [r for r in ALL_RUNS
                if r.get('use_mtl') == use_mtl and r.get('num_labels') == num_labels]
        if not runs:
            print(f'(skip {setup}: no runs)')
            continue

        col_specs = PAPER_COLS[num_labels]
        fmt_rows, num_rows = [], []
        for topic in topics + ['OVERALL']:
            scope_runs = runs if topic == 'OVERALL' else [r for r in runs if r.get('test_topic') == topic]
            if not scope_runs:
                continue
            fmt_row = {'topic': topic}
            num_row = {'topic': topic, 'n_runs': len(scope_runs)}
            for col_name, metric_key in col_specs:
                m, s = _agg(metric_key, scope_runs)
                fmt_row[col_name] = '—' if m is None else f'{m:.4f} ± {s:.4f}'
                num_row[f'{col_name}_mean'] = m
                num_row[f'{col_name}_std']  = s
            fmt_rows.append(fmt_row); num_rows.append(num_row)
            combined_rows.append({'setup': setup, **num_row})

        fmt_df = pd.DataFrame(fmt_rows, columns=['topic', *[c for c, _ in col_specs]])
        num_df = pd.DataFrame(num_rows)

        print(f"\n=== {setup} — Table 4 style ===")
        print(fmt_df.to_string(index=False))

        fmt_path = TABLES_DIR / f'paper_table_{setup}.csv'
        num_path = TABLES_DIR / f'paper_table_{setup}_numeric.csv'
        fmt_df.to_csv(fmt_path, index=False)
        num_df.to_csv(num_path, index=False)
        print(f'  wrote {fmt_path}')
        print(f'  wrote {num_path}')

# Combined long-form CSV across all (setup, scope, metric) tuples.
if combined_rows:
    combined_csv = TABLES_DIR / 'paper_table_all.csv'
    pd.DataFrame(combined_rows).to_csv(combined_csv, index=False)
    print(f'\nwrote {combined_csv}  ({len(combined_rows)} rows)')